## Impor modul yang dibutuhkan

In [14]:
import os
import re
import pandas as pd
from openpyxl import load_workbook

## Evaluasi Provinsi vs Kabupaten/Kota

#### 1. Membaca file Evaluasi.xlsx

In [15]:
# Ganti dengan file Excel kamu
file_path = "data\Evaluasi.xlsx"

# Baca sheet dengan header baris 1-3
df = pd.read_excel(file_path, sheet_name="Prov vs KabKot", header=[0,1,2])

# Gabungkan header multi-level jadi single-level
df.columns = [
    "_".join([str(c).strip() for c in col if "Unnamed" not in str(c)])
        .lower()
        .replace(" ", "_")
    for col in df.columns.values
]

# Hapus double underscore kalau ada
df.columns = [c.replace("__", "_").strip("_") for c in df.columns]

# Cek hasil kolom
print(df.columns.tolist())

# Tampilkan data
print(df.head())


['kode', 'provinsi', 'adhb_pdrb_q1-25', 'adhb_pdrb_q2-25', 'adhb_pkrt_q1-25', 'adhb_pkrt_q2-25', 'adhb_pklnprt_q1-25', 'adhb_pklnprt_q2-25', 'adhb_pkp_q1-25', 'adhb_pkp_q2-25', 'adhb_pmtb_q1-25', 'adhb_pmtb_q2-25', 'adhb_pi_q1-25', 'adhb_pi_q2-25', 'adhb_net_ekspor_q1-25', 'adhb_net_ekspor_q2-25', 'adhk_pdrb_q1-25', 'adhk_pdrb_q2-25', 'adhk_pkrt_q1-25', 'adhk_pkrt_q2-25', 'adhk_pklnprt_q1-25', 'adhk_pklnprt_q2-25', 'adhk_pkp_q1-25', 'adhk_pkp_q2-25', 'adhk_pmtb_q1-25', 'adhk_pmtb_q2-25', 'adhk_pi_q1-25', 'adhk_pi_q2-25', 'adhk_net_ekspor_q1-25', 'adhk_net_ekspor_q2-25', 'distribusi_pdrb_q1-25', 'distribusi_pdrb_q1-25.1', 'distribusi_pdrb_q2-25', 'distribusi_pdrb_q2-25.1', 'distribusi_pkrt_q1-25', 'distribusi_pkrt_q1-25.1', 'distribusi_pkrt_q2-25', 'distribusi_pkrt_q2-25.1', 'distribusi_pklnprt_q1-25', 'distribusi_pklnprt_q1-25.1', 'distribusi_pklnprt_q2-25', 'distribusi_pklnprt_q2-25.1', 'distribusi_pkp_q1-25', 'distribusi_pkp_q1-25.1', 'distribusi_pkp_q2-25', 'distribusi_pkp_q2-25.1',

#### 2. Evaluasi diskrepansi ADHB dan ADHK

In [16]:
# 1. Ambil kolom yang diawali adhb atau adhk
target_cols = [col for col in df.columns if col.startswith("adhb") or col.startswith("adhk")]

# 2. Buat list untuk menampung hasil diskrepansi
evaluasi_diskrepansi = []

# 3. Loop tiap kolom target
for col in target_cols:
    for idx, val in df[col].items():
        if pd.notna(val) and (val < -4.99 or val > 4.99):
            # Pisahkan nama kolom jadi komponen
            # contoh: adhb_pdrb_q1_25 → ['adhb', 'pdrb', 'q1', '25']
            parts = col.split("_")
            
            periode = parts[-1]  # ambil q1 / q2
            record = {
                "kode": df.at[idx, "kode"],
                "provinsi": df.at[idx, "provinsi"],
                "periode": periode,
                "nilai": val,
                "kolom": col,  # tambahan biar tau sumbernya
                "evaluasi": "Diskrepansi lebih dari +- 5 persen"
            }
            evaluasi_diskrepansi.append(record)

# 4. Ubah jadi DataFrame biar enak lihat
evaluasi_diskrepansi = pd.DataFrame(evaluasi_diskrepansi)

print(evaluasi_diskrepansi)

    kode           provinsi periode       nilai                  kolom  \
0     92   Papua Barat Daya   q2-25    5.257010     adhb_pklnprt_q2-25   
1     76     Sulawesi Barat   q1-25    5.072422         adhb_pkp_q1-25   
2     81             Maluku   q1-25    9.297075         adhb_pkp_q1-25   
3     11               Aceh   q1-25  128.940634          adhb_pi_q1-25   
4     15              Jambi   q1-25  -10.189224          adhb_pi_q1-25   
..   ...                ...     ...         ...                    ...   
88    61   Kalimantan Barat   q2-25   73.732307  adhk_net_ekspor_q2-25   
89    62  Kalimantan Tengah   q2-25   -5.621616  adhk_net_ekspor_q2-25   
90    73   Sulawesi Selatan   q2-25  -22.950524  adhk_net_ekspor_q2-25   
91    75          Gorontalo   q2-25  -10.477837  adhk_net_ekspor_q2-25   
92    94              Papua   q2-25   -8.028651  adhk_net_ekspor_q2-25   

                              evaluasi  
0   Diskrepansi lebih dari +- 5 persen  
1   Diskrepansi lebih dari +-

#### 3. Evaluasi distribusi

In [ ]:
# 1. Ambil semua kolom distribusi
dist_cols = [col for col in df.columns if col.startswith("distribusi")]

# 2. Buat base name tanpa suffix `.1`
pairs = {}
for col in dist_cols:
    base = re.sub(r'\.1$', '', col)  # buang .1 di akhir
    pairs.setdefault(base, []).append(col)

# 3. Evaluasi distribusi
evaluasi_dist = []

for base, cols in pairs.items():
    if len(cols) == 2:  # hanya kalau ada pasangan asli + .1
        col0 = [c for c in cols if not c.endswith(".1")][0]
        col1 = [c for c in cols if c.endswith(".1")][0]

        # Ambil periode (contoh: distribusi_pdrb_q1-25 → q1)
        parts = base.split("_")
        periode = parts[-1]  # ambil q1-25, q2-25, dst

        for idx in df.index:
            v0 = df.at[idx, col0]
            v1 = df.at[idx, col1]

            if pd.notna(v0) and pd.notna(v1) and v1 != 0:
                selisih = abs((v0 - v1) / v1)
                if selisih > 0.2:
                    record = {
                        "kode": df.at[idx, "kode"],
                        "provinsi": df.at[idx, "provinsi"],
                        "periode": periode,
                        "nilai": v1,
                        "kolom": col0 + " vs " + col1,
                        "evaluasi": "Distribusi lebih dari +- 20 persen"
                    }
                    evaluasi_dist.append(record)

# 4. Jadi DataFrame
evaluasi_dist = pd.DataFrame(evaluasi_dist)

print(evaluasi_dist)

    kode             provinsi periode      nilai  \
0     11                 Aceh   q1-25   0.878141   
1     18              Lampung   q1-25   0.471715   
2     34        Di Yogyakarta   q1-25  -0.057057   
3     61     Kalimantan Barat   q1-25   2.005087   
4     76       Sulawesi Barat   q1-25  -0.292382   
5     15                Jambi   q2-25   0.393315   
6     18              Lampung   q2-25   0.320699   
7     34        Di Yogyakarta   q2-25   0.801093   
8     61     Kalimantan Barat   q2-25   0.766960   
9     73     Sulawesi Selatan   q2-25   0.260413   
10    11                 Aceh   q1-25  -0.558467   
11    18              Lampung   q1-25  -2.542664   
12    32           Jawa Barat   q1-25   7.654144   
13    52  Nusa Tenggara Barat   q1-25 -20.886224   
14    61     Kalimantan Barat   q1-25   7.135675   
15    11                 Aceh   q2-25  -3.697719   
16    18              Lampung   q2-25   1.063682   
17    21       Kepulauan Riau   q2-25  10.005740   
18    32    

#### 4. Evaluasi growth Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, dan Implisit Y-on-Y 

In [18]:
# 1. Ambil semua kolom Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, dan Implisit Y-on-Y 
growth_cols = [col for col in df.columns if col.startswith("q-to-q") or col.startswith("y-on-y") or col.startswith("c-to-c") or col.startswith("implisit_q-to-q") or col.startswith("implisit_y-on-y")]

# 2. Buat base name tanpa suffix `.1`
pairs = {}
for col in growth_cols:
    base = re.sub(r'\.1$', '', col)  # buang .1 di akhir
    pairs.setdefault(base, []).append(col)

# 3. Evaluasi Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, dan Implisit Y-on-Y
evaluasi_growth = []

for base, cols in pairs.items():
    if len(cols) == 2:  # hanya kalau ada pasangan asli + .1
        col0 = [c for c in cols if not c.endswith(".1")][0]
        col1 = [c for c in cols if c.endswith(".1")][0]

        # Ambil periode (contoh: q-to-q_pdrb_q1-25 → q1-25)
        parts = base.split("_")
        periode = parts[-1]  # ambil q1-25, q2-25, dst

        for idx in df.index:
            v0 = df.at[idx, col0]
            v1 = df.at[idx, col1]

            if pd.notna(v0) and pd.notna(v1) and v1 != 0:
                selisih = abs((v1 - v0) / v0)
                if v0 * v1 < 0:
                    record = {
                        "kode": df.at[idx, "kode"],
                        "provinsi": df.at[idx, "provinsi"],
                        "periode": periode,
                        "nilai": v1,
                        "kolom": col0 + " vs " + col1,
                        "evaluasi": "Growth berlawanan arah"
                    }
                    evaluasi_growth.append(record)
                elif selisih > 2:
                    record = {
                        "kode": df.at[idx, "kode"],
                        "provinsi": df.at[idx, "provinsi"],
                        "periode": periode,
                        "nilai": v1,
                        "kolom": col0 + " vs " + col1,
                        "evaluasi": "Growth lebih dari +- 2 kalinya"
                    }
                    evaluasi_growth.append(record)

# 4. Jadi DataFrame
evaluasi_growth = pd.DataFrame(evaluasi_growth)

print(evaluasi_growth)

     kode          provinsi periode     nilai  \
0      13    Sumatera Barat   q1-25 -0.501638   
1      91       Papua Barat   q2-25 -0.509664   
2      32        Jawa Barat   q1-25  0.954537   
3      65  Kalimantan Utara   q1-25  0.156005   
4      72   Sulawesi Tengah   q1-25  1.100935   
..    ...               ...     ...       ...   
108    73  Sulawesi Selatan   q1-25  0.892260   
109    81            Maluku   q1-25 -0.718580   
110    16  Sumatera Selatan   q2-25  3.928670   
111    32        Jawa Barat   q2-25  2.916537   
112    81            Maluku   q2-25 -0.712034   

                                                 kolom  \
0             q-to-q_pdrb_q1-25 vs q-to-q_pdrb_q1-25.1   
1             q-to-q_pdrb_q2-25 vs q-to-q_pdrb_q2-25.1   
2             q-to-q_pkrt_q1-25 vs q-to-q_pkrt_q1-25.1   
3             q-to-q_pkrt_q1-25 vs q-to-q_pkrt_q1-25.1   
4             q-to-q_pkrt_q1-25 vs q-to-q_pkrt_q1-25.1   
..                                                 ...   
108  

## Menggabungkan seluruh hasil evaluasi

In [ ]:
evaluasi = pd.concat([evaluasi_diskrepansi, evaluasi_dist, evaluasi_growth], ignore_index=True)
print(evaluasi.head())


   kode          provinsi periode       nilai               kolom  \
0    92  Papua Barat Daya   q2-25    5.257010  adhb_pklnprt_q2-25   
1    76    Sulawesi Barat   q1-25    5.072422      adhb_pkp_q1-25   
2    81            Maluku   q1-25    9.297075      adhb_pkp_q1-25   
3    11              Aceh   q1-25  128.940634       adhb_pi_q1-25   
4    15             Jambi   q1-25  -10.189224       adhb_pi_q1-25   

                             evaluasi  
0  Diskrepansi lebih dari +- 5 persen  
1  Diskrepansi lebih dari +- 5 persen  
2  Diskrepansi lebih dari +- 5 persen  
3  Diskrepansi lebih dari +- 5 persen  
4  Diskrepansi lebih dari +- 5 persen  


## Menuliskan hasil evaluasi kedalam file excel

In [20]:
def safe_sheet_name(name):
    return re.sub(r'[:\\/*?\[\]]', "_", str(name))[:31]

for prov in evaluasi["provinsi"].unique():
    df_prov = evaluasi[evaluasi["provinsi"] == prov]
    kode = str(df_prov["kode"].iloc[0])  # ambil kode provinsi

    # Nama file sesuai kode provinsi
    output_file = f"evaluasi_{kode}.xlsx"
    sheet_name = safe_sheet_name(kode + "00")

    if os.path.exists(output_file):
        # Load workbook lama
        book = load_workbook(output_file)

        if sheet_name in book.sheetnames:
            # Sheet sudah ada → cari baris terakhir
            ws = book[sheet_name]
            startrow = ws.max_row  # tulis setelah baris terakhir

            with pd.ExcelWriter(output_file, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
                df_prov.to_excel(writer, sheet_name=sheet_name, index=False, header=False, startrow=startrow)
            print(f"Data {prov} ditambahkan ke sheet {sheet_name} di {output_file}")

        else:
            # Sheet belum ada → tambahkan sheet baru
            with pd.ExcelWriter(output_file, engine="openpyxl", mode="a") as writer:
                df_prov.to_excel(writer, sheet_name=sheet_name, index=False)
            print(f"Sheet {sheet_name} ditambahkan ke {output_file}")

    else:
        # File belum ada → buat baru
        with pd.ExcelWriter(output_file, engine="openpyxl", mode="w") as writer:
            df_prov.to_excel(writer, sheet_name=sheet_name, index=False)
        print(f"File baru dibuat: {output_file}, dengan sheet {sheet_name}")


File baru dibuat: evaluasi_92.xlsx, dengan sheet 9200
File baru dibuat: evaluasi_76.xlsx, dengan sheet 7600
File baru dibuat: evaluasi_81.xlsx, dengan sheet 8100
File baru dibuat: evaluasi_11.xlsx, dengan sheet 1100
File baru dibuat: evaluasi_15.xlsx, dengan sheet 1500
File baru dibuat: evaluasi_18.xlsx, dengan sheet 1800
File baru dibuat: evaluasi_34.xlsx, dengan sheet 3400
File baru dibuat: evaluasi_61.xlsx, dengan sheet 6100
File baru dibuat: evaluasi_73.xlsx, dengan sheet 7300
File baru dibuat: evaluasi_91.xlsx, dengan sheet 9100
File baru dibuat: evaluasi_64.xlsx, dengan sheet 6400
File baru dibuat: evaluasi_13.xlsx, dengan sheet 1300
File baru dibuat: evaluasi_14.xlsx, dengan sheet 1400
File baru dibuat: evaluasi_16.xlsx, dengan sheet 1600
File baru dibuat: evaluasi_31.xlsx, dengan sheet 3100
File baru dibuat: evaluasi_32.xlsx, dengan sheet 3200
File baru dibuat: evaluasi_33.xlsx, dengan sheet 3300
File baru dibuat: evaluasi_62.xlsx, dengan sheet 6200
File baru dibuat: evaluasi_7

## Evaluasi Growth dan Revisi

#### 1. Membaca file Evaluasi.xlsx pada sheet Growth & Revisi

In [21]:
# Ganti dengan file Excel kamu
file_path = "data\Evaluasi.xlsx"

# Baca sheet dengan header baris 1-3
df = pd.read_excel(file_path, sheet_name="Growth & Revisi", header=[0,1,2])

# Gabungkan header multi-level jadi single-level
df.columns = [
    "_".join([str(c).strip() for c in col if "Unnamed" not in str(c)])
        .lower()
        .replace(" ", "_")
    for col in df.columns.values
]

# Hapus double underscore kalau ada
df.columns = [c.replace("__", "_").strip("_") for c in df.columns]

# Cek hasil kolom
print(df.columns.tolist())

# Tampilkan data
print(df.head())


['kode', 'kabupaten/kota', 'adhb_pdrb_q1-25_ril', 'adhb_pdrb_q1-25_rev', 'adhb_pdrb_q2-25', 'adhb_pkrt_q1-25_ril', 'adhb_pkrt_q1-25_rev', 'adhb_pkrt_q2-25', 'adhb_pklnprt_q1-25_ril', 'adhb_pklnprt_q1-25_rev', 'adhb_pklnprt_q2-25', 'adhb_pkp_q1-25_ril', 'adhb_pkp_q1-25_rev', 'adhb_pkp_q2-25', 'adhb_pmtb_q1-25_ril', 'adhb_pmtb_q1-25_rev', 'adhb_pmtb_q2-25', 'adhb_pi_q1-25_ril', 'adhb_pi_q1-25_rev', 'adhb_pi_q2-25', 'adhb_net_ekspor_q1-25_ril', 'adhb_net_ekspor_q1-25_rev', 'adhb_net_ekspor_q2-25', 'adhk_pdrb_q1-25_ril', 'adhk_pdrb_q1-25_rev', 'adhk_pdrb_q2-25', 'adhk_pkrt_q1-25_ril', 'adhk_pkrt_q1-25_rev', 'adhk_pkrt_q2-25', 'adhk_pklnprt_q1-25_ril', 'adhk_pklnprt_q1-25_rev', 'adhk_pklnprt_q2-25', 'adhk_pkp_q1-25_ril', 'adhk_pkp_q1-25_rev', 'adhk_pkp_q2-25', 'adhk_pmtb_q1-25_ril', 'adhk_pmtb_q1-25_rev', 'adhk_pmtb_q2-25', 'adhk_pi_q1-25_ril', 'adhk_pi_q1-25_rev', 'adhk_pi_q2-25', 'adhk_net_ekspor_q1-25_ril', 'adhk_net_ekspor_q1-25_rev', 'adhk_net_ekspor_q2-25', 'distribusi_pdrb_q1-25_ril'

#### 2. Evaluasi revisi ADHB dan ADHK

In [ ]:
# 1. Ambil semua kolom ADHB dan ADHK 
revisi_harga_cols = [col for col in df.columns if col.startswith("adhb") or col.startswith("adhk")]

# 2. Buat base name tanpa suffix `ril/rev`
pairs = {}
for col in revisi_harga_cols:
    if "ril" in col or "rev" in col:
        base = "_".join(col.split("_")[:-1])  # buang suffix ril/rev
        pairs.setdefault(base, []).append(col)

# 3. Evaluasi ADHB dan ADHK
evaluasi_revisi_harga = []

for base, cols in pairs.items():
    if len(cols) == 2:  # hanya kalau ada pasangan asli + .1
        col0 = [c for c in cols if c.endswith("_ril")][0]
        col1 = [c for c in cols if c.endswith("_rev")][0]

        # Ambil periode (contoh: q-to-q_pdrb_q1-25 → q1-25)
        parts = base.split("_")
        periode = parts[-1]  # ambil q1-25, q2-25, dst

        for idx in df.index:
            v0 = df.at[idx, col0]
            v1 = df.at[idx, col1]

            if pd.notna(v0) and pd.notna(v1) and v1 != 0:
                selisih = abs((v1 - v0) / v0)
                if v0 * v1 < 0:
                    record = {
                        "kode": df.at[idx, "kode"],
                        "kabupaten/kota": df.at[idx, "kabupaten/kota"],
                        "periode": periode,
                        "nilai": v1,
                        "kolom": col0 + " vs " + col1,
                        "evaluasi": "Angka revisi berlawanan arah"
                    }
                    evaluasi_revisi_harga.append(record)
                elif selisih > 0.1:
                    record = {
                        "kode": df.at[idx, "kode"],
                        "kabupaten/kota": df.at[idx, "kabupaten/kota"],
                        "periode": periode,
                        "nilai": v1,
                        "kolom": col0 + " vs " + col1,
                        "evaluasi": "Angka revisi lebih dari +- 10 persen"
                    }
                    evaluasi_revisi_harga.append(record)

# 4. Jadi DataFrame
evaluasi_revisi_harga = pd.DataFrame(evaluasi_revisi_harga)

print(evaluasi_revisi_harga)

     kode                kabupaten/kota periode          nilai  \
0    8172                     Kota Tual   q1-25  453419.939725   
1    8172                     Kota Tual   q1-25  262787.666764   
2    8172                     Kota Tual   q1-25   11269.647978   
3    9606          Kabupaten Intan Jaya   q1-25    2714.899320   
4    2101             Kabupaten Karimun   q1-25  243385.795100   
..    ...                           ...     ...            ...   
214  7604              Kabupaten Mamuju   q1-25  158507.071628   
215  7606       Kabupaten Mamuju Tengah   q1-25   14200.921424   
216  8107  Kabupaten Seram Bagian Timur   q1-25 -477664.519005   
217  8201     Kabupaten Halmahera Barat   q1-25   -3950.116295   
218  9604              Kabupaten Nabire   q1-25  -12038.372220   

                                                 kolom  \
0           adhb_pdrb_q1-25_ril vs adhb_pdrb_q1-25_rev   
1           adhb_pkrt_q1-25_ril vs adhb_pkrt_q1-25_rev   
2     adhb_pklnprt_q1-25_ril vs a

#### 3. Evaluasi revisi distribusi

In [33]:
# =OR(ABS((AW4-AV4)/AV4)>0.1, AV4*AW4<0) pkrt 
# =OR(ABS((AZ4-AY4)/AY4)>0.1, AY4*AZ4<0) pklnprt
# =OR(ABS((BC4-BB4)/BB4)>0.1, BB4*BC4<0) pkp
# =OR(ABS((BF4-BE4)/BE4)>0.1, BE4*BF4<0) pmtb
# =BH4*BI4<0 pi
# =BK4*BL4<0 net ekspor 

# 1. Ambil semua kolom distribusi 
revisi_dist_cols = [col for col in df.columns if col.startswith("distribusi")]

# 2. Buat base name tanpa suffix `ril/rev`
pairs = {}
for col in revisi_dist_cols:
    if "ril" in col or "rev" in col:
        base = "_".join(col.split("_")[:-1])  # buang suffix ril/rev
        pairs.setdefault(base, []).append(col)

# 3. Evaluasi distribusi
evaluasi_revisi_dist = []

for base, cols in pairs.items():
    if len(cols) == 2:  # hanya kalau ada pasangan asli + .1
        col0 = [c for c in cols if c.endswith("_ril")][0]
        col1 = [c for c in cols if c.endswith("_rev")][0]

        # Ambil periode (contoh: q-to-q_pdrb_q1-25 → q1-25)
        parts = base.split("_")
        periode = parts[-1]  # ambil q1-25, q2-25, dst

        for idx in df.index:
            v0 = df.at[idx, col0]
            v1 = df.at[idx, col1]

            if pd.notna(v0) and pd.notna(v1) and v1 != 0:
                selisih = abs((v1 - v0) / v0)
                if v0 * v1 < 0:
                    record = {
                        "kode": df.at[idx, "kode"],
                        "kabupaten/kota": df.at[idx, "kabupaten/kota"],
                        "periode": periode,
                        "nilai": v1,
                        "kolom": col0 + " vs " + col1,
                        "evaluasi": "Distribusi revisi berlawanan arah"
                    }
                    evaluasi_revisi_dist.append(record)
                elif selisih > 0.1 and "pi" not in col1 and "net_ekspor" not in col1:
                    record = {
                        "kode": df.at[idx, "kode"],
                        "kabupaten/kota": df.at[idx, "kabupaten/kota"],
                        "periode": periode,
                        "nilai": v1,
                        "kolom": col0 + " vs " + col1,
                        "evaluasi": "Distribusi revisi lebih dari +- 10 persen"
                    }
                    evaluasi_revisi_dist.append(record)

# 4. Jadi DataFrame
evaluasi_revisi_dist = pd.DataFrame(evaluasi_revisi_dist)

print(evaluasi_revisi_dist)

    kode                       kabupaten/kota periode     nilai  \
0   9606                 Kabupaten Intan Jaya   q1-25  0.721365   
1   2101                    Kabupaten Karimun   q1-25  5.207861   
2   7317                       Kabupaten Luwu   q1-25  4.591734   
3   1805             Kabupaten Lampung Tengah   q1-25 -1.191955   
4   1812        Kabupaten Tulang Bawang Barat   q1-25  1.058070   
5   2102                     Kabupaten Bintan   q1-25 -1.298204   
6   5301                Kabupaten Sumba Barat   q1-25 -0.129834   
7   5302                Kabupaten Sumba Timur   q1-25 -1.938663   
8   5303                     Kabupaten Kupang   q1-25 -0.076272   
9   5304       Kabupaten Timor Tengah Selatan   q1-25 -1.736143   
10  5305         Kabupaten Timor Tengah Utara   q1-25 -2.992728   
11  5306                       Kabupaten Belu   q1-25 -3.867443   
12  5307                       Kabupaten Alor   q1-25 -1.502214   
13  5308                    Kabupaten Lembata   q1-25 -0.89025

## Menggabungkan seluruh hasil evaluasi

In [34]:
evaluasi = pd.concat([evaluasi_revisi_harga, evaluasi_revisi_dist], ignore_index=True)
print(evaluasi.head())


   kode        kabupaten/kota periode          nilai  \
0  8172             Kota Tual   q1-25  453419.939725   
1  8172             Kota Tual   q1-25  262787.666764   
2  8172             Kota Tual   q1-25   11269.647978   
3  9606  Kabupaten Intan Jaya   q1-25    2714.899320   
4  2101     Kabupaten Karimun   q1-25  243385.795100   

                                              kolom  \
0        adhb_pdrb_q1-25_ril vs adhb_pdrb_q1-25_rev   
1        adhb_pkrt_q1-25_ril vs adhb_pkrt_q1-25_rev   
2  adhb_pklnprt_q1-25_ril vs adhb_pklnprt_q1-25_rev   
3  adhb_pklnprt_q1-25_ril vs adhb_pklnprt_q1-25_rev   
4          adhb_pkp_q1-25_ril vs adhb_pkp_q1-25_rev   

                         evaluasi  
0  Growth lebih dari +- 2 kalinya  
1  Growth lebih dari +- 2 kalinya  
2  Growth lebih dari +- 2 kalinya  
3  Growth lebih dari +- 2 kalinya  
4  Growth lebih dari +- 2 kalinya  


## Menuliskan hasil evaluasi kedalam file excel

In [35]:
def safe_sheet_name(name):
    return re.sub(r'[:\\/*?\[\]]', "_", str(name))[:31]

for kabkot in evaluasi["kabupaten/kota"].unique():
    df_kabkot = evaluasi[evaluasi["kabupaten/kota"] == kabkot]
    kode = str(df_kabkot["kode"].iloc[0])  # ambil kode kabupaten/kota

    # Nama file sesuai kode kabupaten/kota
    output_file = f"evaluasi_{kode[:2]}.xlsx"
    sheet_name = safe_sheet_name(kode)

    if os.path.exists(output_file):
        # Load workbook lama
        book = load_workbook(output_file)

        if sheet_name in book.sheetnames:
            # Sheet sudah ada → cari baris terakhir
            ws = book[sheet_name]
            startrow = ws.max_row  # tulis setelah baris terakhir

            with pd.ExcelWriter(output_file, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
                df_kabkot.to_excel(writer, sheet_name=sheet_name, index=False, header=False, startrow=startrow)
            print(f"Data {kabkot} ditambahkan ke sheet {sheet_name} di {output_file}")

        else:
            # Sheet belum ada → tambahkan sheet baru
            with pd.ExcelWriter(output_file, engine="openpyxl", mode="a") as writer:
                df_kabkot.to_excel(writer, sheet_name=sheet_name, index=False)
            print(f"Sheet {sheet_name} ditambahkan ke {output_file}")

    else:
        # File belum ada → buat baru
        with pd.ExcelWriter(output_file, engine="openpyxl", mode="w") as writer:
            df_kabkot.to_excel(writer, sheet_name=sheet_name, index=False)
        print(f"File baru dibuat: {output_file}, dengan sheet {sheet_name}")


File baru dibuat: evaluasi_81.xlsx, dengan sheet 8172
File baru dibuat: evaluasi_96.xlsx, dengan sheet 9606
File baru dibuat: evaluasi_21.xlsx, dengan sheet 2101
File baru dibuat: evaluasi_73.xlsx, dengan sheet 7317
File baru dibuat: evaluasi_11.xlsx, dengan sheet 1172
Sheet 1174 ditambahkan ke evaluasi_11.xlsx
File baru dibuat: evaluasi_12.xlsx, dengan sheet 1225
Sheet 1278 ditambahkan ke evaluasi_12.xlsx
File baru dibuat: evaluasi_15.xlsx, dengan sheet 1507
File baru dibuat: evaluasi_18.xlsx, dengan sheet 1804
Sheet 1805 ditambahkan ke evaluasi_18.xlsx
Sheet 1809 ditambahkan ke evaluasi_18.xlsx
Sheet 1812 ditambahkan ke evaluasi_18.xlsx
Sheet 1872 ditambahkan ke evaluasi_18.xlsx
File baru dibuat: evaluasi_19.xlsx, dengan sheet 1904
Sheet 2102 ditambahkan ke evaluasi_21.xlsx
Sheet 2171 ditambahkan ke evaluasi_21.xlsx
File baru dibuat: evaluasi_33.xlsx, dengan sheet 3313
File baru dibuat: evaluasi_36.xlsx, dengan sheet 3603
Sheet 3673 ditambahkan ke evaluasi_36.xlsx
File baru dibuat: e